# CS-781 Phase 2 (L-NER) — Step 0: data + tokenizer alignment check (no training, CPU is enough)

**What this notebook does** (≈5–10 min, no GPU needed — set *Accelerator: None*):
1. Downloads the Phase-2 data (`Phase_2_LNER/`) and **your Phase-1 BPE tokenizer** from the bucket `kavincg23/cs781-models`.
2. Checks the data format (offsets, labels, overlaps, the empty test document).
3. Builds **character offsets for every BPE token** and proves the token ids are *identical* to `BPETokenizer.encode` (so the Phase-1 models see exactly what they were trained on).
4. Measures the **strict-F1 ceiling** of token-level tagging: if a model predicted every token's tag perfectly, what strict macro-F1 would it get? Three ways of feeding text to the tokenizer are compared, each with and without a punctuation-trimming repair learned on the *other* fold.
5. Counts tokens and sliding windows per document.
6. Runs the **official `evaluate_phase2.py`** on gold-vs-gold (must be 1.0) and on the oracle predictions (must equal our own scorer).
7. Saves `phase2_alignment_report.json` and uploads it to the bucket (for the report).

Nothing here trains or submits anything. The numbers it prints decide one design choice for all later notebooks (see the last cell).

Secrets needed (same as Phase 1): **`HF`** (read, gated dataset) and **`HF_WRITE`** (bucket).

## 1. Settings

In [ ]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P2_DIR = "Phase_2_LNER"
TRAIN_FILES = {f"fold{i}": (f"{P2_DIR}/Train/fold{i}-text.json", f"{P2_DIR}/Train/fold{i}-gold.json") for i in (1, 2)}
LABELS_FILE = f"{P2_DIR}/Train/labels.txt"
TEST_FILE = f"{P2_DIR}/Test/fold3-text.json"

HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"   # shared by every Phase-1 model
WORK = "/kaggle/working"

# window sizes to report (tokens); the Phase-1 models were trained on 2048-token chunks
WINDOW_SETTINGS = [(2048, 1024), (2048, 1536), (1024, 512), (512, 256)]
REPORT_PATH = f"{WORK}/phase2_alignment_report.json"


## 2. Install, Hugging Face login, bucket helper

In [ ]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time
from collections import Counter, defaultdict
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
    print(f"Uploaded {name} to bucket {HF_BUCKET}")

print("Logged in; bucket:", HF_BUCKET)


## 3. Load the Phase-2 data

In [ ]:
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

LABELS = [l.strip() for l in open(get_dataset_file(LABELS_FILE), encoding="utf-8") if l.strip()]
TEXTS, GOLDS = {}, {}
for fold, (tf, gf) in TRAIN_FILES.items():
    TEXTS[fold] = load_json(get_dataset_file(tf))
    GOLDS[fold] = load_json(get_dataset_file(gf))
TEST_TEXTS = load_json(get_dataset_file(TEST_FILE))

print("labels:", LABELS)
for fold in TEXTS:
    n_ent = sum(len(v) for v in GOLDS[fold].values())
    print(f"{fold}: {len(TEXTS[fold])} docs, {n_ent} entities")
print(f"test: {len(TEST_TEXTS)} docs")


## 4. Data format checks

In [ ]:
report = {"data": {}}
for fold in TEXTS:
    T, G = TEXTS[fold], GOLDS[fold]
    assert set(T) == set(G), f"{fold}: text/gold ids differ"
    cnt, mismatch, overlaps, ws_edge = Counter(), 0, 0, 0
    for did, ents in G.items():
        t = T[did]
        for ent in ents:
            s, e, lab = ent[0], ent[1], ent[2]
            assert lab in LABELS, lab
            cnt[lab] += 1
            if len(ent) > 3 and t[s:e] != ent[3]:
                mismatch += 1
            if t[s:e] != t[s:e].strip():
                ws_edge += 1
        es = sorted((x[0], x[1]) for x in ents)
        overlaps += sum(1 for (a, b), (c, d) in zip(es, es[1:]) if c < b)
    words = sorted(len(t.split()) for t in T.values())
    report["data"][fold] = dict(docs=len(T), entities=sum(cnt.values()), per_label=dict(cnt), text_offset_mismatch=mismatch,
                                overlapping_pairs=overlaps, spans_with_edge_whitespace=ws_edge,
                                words_min=words[0], words_median=words[len(words)//2], words_max=words[-1])
    print(fold, json.dumps(report["data"][fold]))
empty_test = [k for k, v in TEST_TEXTS.items() if not v.strip()]
report["data"]["test_empty_docs"] = empty_test
print("test docs with no text (must still be in prediction.json as []):", empty_test)
print("\n-> end offsets are EXCLUSIVE if text_offset_mismatch == 0; overlapping_pairs == 0 means flat BIO tagging can represent every entity.")


## 5. Your Phase-1 BPE tokenizer (class copied unchanged from the Phase-1 notebooks) — loaded from the bucket

In [ ]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)




tok_path = fetch_model_file(TOKENIZER_NAME)
bpe_tokenizer = BPETokenizer.load(tok_path)
print("tokenizer:", TOKENIZER_NAME, "| vocab", bpe_tokenizer.get_vocab_size(), "| merges", len(bpe_tokenizer.merges))


## 6. Offset-aware encoding, BIO tags, span decoding, strict scorer

`encode_with_offsets(tok, text, mode)` returns the token ids **and** each token's `(start_char, end_char)` in the raw text.
- `mode="none"`: exactly the Phase-1 tokenization (ids identical to `bpe_tokenizer.encode`).
- `mode="punct"`: each whitespace word is first split into letter/digit runs and single punctuation characters, then BPE-encoded. This changes the ids versus pretraining (e.g. `Haryana,` becomes `Haryana</w>` + `,</w>`) — a trade-off to measure, not a free fix.
- `mode="punct_digit"`: also splits letters from digits (`Pradesh1` -> `Pradesh`, `1`).

Boundary repair: after decoding token spans back to characters, a table learned **only on the training fold** says, for each (label, trailing/leading punctuation char), whether gold entities usually exclude that character; if so it is trimmed.

In [ ]:
# ===== Phase 2 L-NER: character offsets for the Phase-1 BPE tokenizer, BIO tags, span decoding =====
# Ancillary pipeline code (data handling / alignment / decoding). It does NOT change the tokenizer: the token ids
# produced here are exactly BPETokenizer.encode(text) (checked on every document by check_encode_identical).

LABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
_END_MARK = "</w>"


def iter_words(text):
    """(start_char, end_char) of every whitespace-separated word, using exactly str.split()'s notion of whitespace
    (str.isspace), so the words are the same ones BPETokenizer.encode() sees."""
    n, i = len(text), 0
    while i < n:
        while i < n and text[i].isspace():
            i += 1
        if i >= n:
            break
        j = i
        while j < n and not text[j].isspace():
            j += 1
        yield i, j
        i = j


import re as _re
PIECE_MODES = {
    "none": None,                                                   # exactly the Phase-1 tokenization
    "punct": _re.compile(r"\w+|[^\w\s]"),                          # split every punctuation char off
    "punct_digit": _re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]"),       # also split letters from digits
}


def iter_pieces(text, mode="none"):
    """Words as the tokenizer sees them. mode 'none' = whitespace words (Phase-1 behaviour); the other modes split
    each whitespace word further, which changes the token ids versus pretraining (a design choice to measure)."""
    rx = PIECE_MODES[mode]
    for ws, we in iter_words(text):
        if rx is None:
            yield ws, we
        else:
            for m in rx.finditer(text, ws, we):
                yield m.start(), m.end()


def encode_with_offsets(tok, text, mode="none"):
    """Return (ids, spans): ids == tok.encode(text); spans[k] = (start_char, end_char) of token k in `text`
    (end exclusive). A token's symbols are UTF-8 bytes held as chr(b); when a token boundary falls inside a
    multi-byte character, that character is given to the token holding its first byte (the next token then starts
    after it; if a token holds only continuation bytes it gets an empty span at that position)."""
    unk = tok.stoi["<unk>"]
    ids, spans = [], []
    for ws, we in iter_pieces(text, mode):
        word = text[ws:we]
        syms = tok._tokenize_word(word)
        cached = tok._id_cache.get(word)
        if cached is None:
            cached = [tok.stoi.get(t, unk) for t in syms]
            tok._id_cache[word] = cached
        # byte offset -> char offset inside the word
        byte_start_of_char = []
        b = 0
        for ch in word:
            byte_start_of_char.append(b)
            b += len(ch.encode("utf-8"))
        total_bytes = b
        byte_to_char = [0] * (total_bytes + 1)
        for ci, bs in enumerate(byte_start_of_char):
            nb = (byte_start_of_char[ci + 1] if ci + 1 < len(byte_start_of_char) else total_bytes) - bs
            for k in range(nb):
                byte_to_char[bs + k] = ci if k == 0 else ci + 1      # a continuation byte "belongs" after its char start
        byte_to_char[total_bytes] = len(word)
        bpos = 0
        for sym in syms:
            core = sym[:-len(_END_MARK)] if sym.endswith(_END_MARK) else sym
            nbytes = len(core)                                       # each symbol char is one byte
            cs = byte_to_char[bpos] if bpos < total_bytes else len(word)
            bpos += nbytes
            ce = byte_to_char[bpos] if bpos < total_bytes else len(word)
            if ce < cs:
                ce = cs
            spans.append((ws + cs, ws + ce))
        ids.extend(cached)
    return ids, spans


def check_encode_identical(tok, docs):
    """Assert our ids equal tok.encode on every doc and that spans are monotone and inside the text."""
    for did, text in docs.items():
        ids, spans = encode_with_offsets(tok, text)
        ref = tok.encode(text)
        assert ids == ref, f"doc {did}: ids differ from BPETokenizer.encode"
        prev = 0
        for (s, e) in spans:
            assert prev <= s <= e <= len(text), f"doc {did}: bad span {(s, e)}"
            prev = e
    return True


# ---------------- BIO tags ----------------
IGNORE = -100        # label for tokens that cover no character (loss ignores them; decoding skips them)
def bio_tag_names(labels):
    return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]


def gold_to_token_tags(spans, ents, labels):
    """Token tag ids (0 = O, 1+2*li = B-label, 2+2*li = I-label). A token is inside an entity if its char span
    overlaps the entity. Also returns how many entities have a start/end that is NOT on a token boundary."""
    tag = [0 if e > s else IGNORE for s, e in spans]                # empty-span tokens (e.g. a lone '</w>') -> ignored
    lab_idx = {l: i for i, l in enumerate(labels)}
    starts = {s for s, e in spans if e > s}
    ends = {e for s, e in spans if e > s}
    misaligned = []
    ents_sorted = sorted(ents, key=lambda x: (x[0], x[1]))
    k = 0
    for ent in ents_sorted:
        s, e, lab = ent[0], ent[1], ent[2]
        li = lab_idx[lab]
        if s not in starts or e not in ends:
            misaligned.append(ent)
        while k < len(spans) and spans[k][1] <= s:
            k += 1
        j, first = k, True
        while j < len(spans) and spans[j][0] < e:
            if spans[j][1] > spans[j][0]:                            # skip empty (continuation-byte) tokens
                if tag[j] == 0:                                     # never overwrite another entity's token
                    tag[j] = 1 + 2 * li if first else 2 + 2 * li
                first = False
            j += 1
    return tag, misaligned


def token_tags_to_spans(spans, tag, labels):
    """BIO tag ids -> [(start_char, end_char, label)] with lenient repair: an I- tag that does not continue an
    entity of the same label starts a new one."""
    out, cur = [], None
    for (s, e), t in zip(spans, tag):
        if e <= s or t == IGNORE:
            continue
        if t == 0:
            if cur:
                out.append(tuple(cur)); cur = None
            continue
        li, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        lab = labels[li]
        if cur and not is_b and cur[2] == lab:
            cur[1] = e
        else:
            if cur:
                out.append(tuple(cur))
            cur = [s, e, lab]
    if cur:
        out.append(tuple(cur))
    return out


# ---------------- boundary repair (character-level, after token decoding) ----------------
TRAIL_PUNCT = set(",.;:)]}'\"’”`-/")
LEAD_PUNCT = set("([{'\"‘“`-:./")


def learn_trim_table(docs, golds, tok_spans_by_doc, labels):
    """Data-driven boundary repair learned on TRAINING docs only. For each (label, trailing char) seen at the end of
    an oracle token span, count how often the gold entity excludes that char -> trim if excluded > kept.
    Same for leading chars. Returns (trail_rule, lead_rule) dicts {(label, ch): True/False}."""
    from collections import Counter
    t_ex, t_keep, l_ex, l_keep = Counter(), Counter(), Counter(), Counter()
    for did, ents in golds.items():
        text, spans = docs[did], tok_spans_by_doc[did]
        tag, _ = gold_to_token_tags(spans, ents, labels)
        pred = token_tags_to_spans(spans, tag, labels)
        # match each oracle span to the gold entity it covers (same label, contained)
        gl = sorted((e[0], e[1], e[2]) for e in ents)
        for (s, e, lab) in pred:
            cands = [g for g in gl if g[2] == lab and g[0] >= s and g[1] <= e]
            if len(cands) != 1:
                continue
            gs, ge, _ = cands[0]
            # trailing chars of the token span beyond gold end
            for p in range(e - 1, s - 1, -1):
                ch = text[p]
                if ch not in TRAIL_PUNCT:
                    break
                if p >= ge:
                    t_ex[(lab, ch)] += 1
                else:
                    t_keep[(lab, ch)] += 1
                    break
            for p in range(s, e):
                ch = text[p]
                if ch not in LEAD_PUNCT:
                    break
                if p < gs:
                    l_ex[(lab, ch)] += 1
                else:
                    l_keep[(lab, ch)] += 1
                    break
    trail = {k: t_ex[k] > t_keep[k] for k in set(t_ex) | set(t_keep)}
    lead = {k: l_ex[k] > l_keep[k] for k in set(l_ex) | set(l_keep)}
    return trail, lead


def repair_span(text, s, e, lab, trail_rule, lead_rule):
    while e > s + 1 and text[e - 1] in TRAIL_PUNCT and trail_rule.get((lab, text[e - 1]), False):
        e -= 1
    while s < e - 1 and text[s] in LEAD_PUNCT and lead_rule.get((lab, text[s]), False):
        s += 1
    return s, e


def finalize_doc_entities(text, raw_spans, trail_rule=None, lead_rule=None):
    """Repair, drop empty/invalid, de-duplicate (the evaluator FAILS on duplicate (start,end,label)), sort."""
    seen, out = set(), []
    for s, e, lab in raw_spans:
        if trail_rule is not None:
            s, e = repair_span(text, s, e, lab, trail_rule, lead_rule or {})
        s, e = int(s), int(e)
        if not (0 <= s < e <= len(text)):
            continue
        key = (s, e, lab)
        if key in seen:
            continue
        seen.add(key)
        out.append([s, e, lab])
    out.sort()
    return out


# ---------------- strict scorer (same arithmetic as evaluate_phase2.py) + error breakdown ----------------
def strict_scores(golds, preds, labels):
    stats = {l: [0, 0, 0] for l in labels}
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        ps = {(x[0], x[1], x[2]) for x in preds.get(did, [])}
        for l in labels:
            a = {x for x in gs if x[2] == l}; b = {x for x in ps if x[2] == l}
            stats[l][0] += len(a & b); stats[l][1] += len(b - a); stats[l][2] += len(a - b)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[l] = dict(P=p, R=r, F1=f, tp=tp, fp=fp, fn=fn)
    n = len(labels)
    return dict(macro_F1=sum(v["F1"] for v in per.values()) / n,
                macro_P=sum(v["P"] for v in per.values()) / n,
                macro_R=sum(v["R"] for v in per.values()) / n, per_label=per)


def error_breakdown(golds, preds):
    """Classify each wrong prediction: boundary error (overlaps a gold entity of the same label), label error (same
    exact span, other label), boundary+label, or spurious (overlaps no gold entity). Also gold entities missed."""
    from collections import Counter
    c = Counter()
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        for (s, e, l) in {(x[0], x[1], x[2]) for x in preds.get(did, [])}:
            if (s, e, l) in gs:
                c["correct"] += 1; continue
            same_span = [x for x in gs if x[0] == s and x[1] == e]
            over = [x for x in gs if x[0] < e and s < x[1]]
            if same_span:
                c["label_error"] += 1
            elif any(x[2] == l for x in over):
                c["boundary_error"] += 1
            elif over:
                c["boundary_and_label_error"] += 1
            else:
                c["spurious"] += 1
        ps = {(x[0], x[1]) for x in preds.get(did, [])}
        for x in gs:
            if not any(p[0] < x[1] and x[0] < p[1] for p in ps):
                c["missed_entirely"] += 1
    return dict(c)


# ---------------- sliding windows ----------------
def make_windows(n_tokens, window, stride):
    """Start indices of windows of length `window` every `stride` tokens, the last one flush with the end."""
    if n_tokens <= window:
        return [0]
    starts = list(range(0, n_tokens - window + 1, stride))
    if starts[-1] + window < n_tokens:
        starts.append(n_tokens - window)
    return starts


## 7. Token ids are identical to `BPETokenizer.encode` (mode 'none') on every document

In [ ]:
t0 = time.time()
all_docs = {f"{fold}:{k}": v for fold in TEXTS for k, v in TEXTS[fold].items()}
all_docs.update({f"test:{k}": v for k, v in TEST_TEXTS.items()})
check_encode_identical(bpe_tokenizer, all_docs)
print(f"OK: ids identical to BPETokenizer.encode on all {len(all_docs)} documents; spans monotone and inside the text ({time.time()-t0:.0f}s)")


## 8. Strict-F1 ceiling of token-level tagging (the 'oracle')
For every gold entity: does it start and end exactly on token boundaries? Then gold → token BIO tags → back to character spans → strict score. Repair is learned on the *other* fold (2-fold, no peeking).

In [ ]:
MODES = ["none", "punct", "punct_digit"]
SPANS = {m: {fold: {} for fold in TEXTS} for m in MODES}
NTOK = {m: {} for m in MODES}
for m in MODES:
    for fold in TEXTS:
        for did, t in TEXTS[fold].items():
            ids, sp = encode_with_offsets(bpe_tokenizer, t, m)
            SPANS[m][fold][did] = sp
            NTOK[m][f"{fold}:{did}"] = len(ids)
    for did, t in TEST_TEXTS.items():
        NTOK[m][f"test:{did}"] = len(encode_with_offsets(bpe_tokenizer, t, m)[0])

report["oracle"] = {}
other = {"fold1": "fold2", "fold2": "fold1"}
for m in MODES:
    report["oracle"][m] = {}
    for fold in TEXTS:
        T, G, S = TEXTS[fold], GOLDS[fold], SPANS[m][fold]
        mis, tot = Counter(), Counter()
        pred_raw, pred_rep = {}, {}
        tr = other[fold]
        trail, lead = learn_trim_table(TEXTS[tr], GOLDS[tr], SPANS[m][tr], LABELS)
        for did in G:
            tag, bad = gold_to_token_tags(S[did], G[did], LABELS)
            for e in bad: mis[e[2]] += 1
            for e in G[did]: tot[e[2]] += 1
            raw = token_tags_to_spans(S[did], tag, LABELS)
            pred_raw[did] = finalize_doc_entities(T[did], raw)
            pred_rep[did] = finalize_doc_entities(T[did], raw, trail, lead)
        s_raw, s_rep = strict_scores(G, pred_raw, LABELS), strict_scores(G, pred_rep, LABELS)
        n_words = sum(len(t.split()) for t in T.values())
        n_tok = sum(NTOK[m][f"{fold}:{d}"] for d in T)
        report["oracle"][m][fold] = dict(
            misaligned=sum(mis.values()), entities=sum(tot.values()), misaligned_per_label=dict(mis),
            tokens_per_word=round(n_tok / n_words, 4),
            ceiling_macroF1_no_repair=round(s_raw["macro_F1"], 4), ceiling_macroF1_with_repair=round(s_rep["macro_F1"], 4),
            ceiling_F1_per_label_with_repair={l: round(v["F1"], 4) for l, v in s_rep["per_label"].items()})
        if m == "none" and fold == "fold1":
            ORACLE_FOLD1_NONE = (pred_raw, s_raw)

print(f"{'mode':12s} {'fold':6s} {'tok/word':>8s} {'misaligned':>14s} {'ceiling F1 raw':>15s} {'ceiling F1 +repair':>19s}")
for m in MODES:
    for fold in TEXTS:
        r = report["oracle"][m][fold]
        print(f"{m:12s} {fold:6s} {r['tokens_per_word']:8.3f} {r['misaligned']:6d}/{r['entities']:<6d} "
              f"{r['ceiling_macroF1_no_repair']:15.4f} {r['ceiling_macroF1_with_repair']:19.4f}")
print("\nper-label ceiling with repair, mode 'none':")
for fold in TEXTS:
    print(" ", fold, report["oracle"]["none"][fold]["ceiling_F1_per_label_with_repair"])


### Examples of entities that do not fall on token boundaries (mode 'none')

In [ ]:
shown = 0
T, G, S = TEXTS["fold1"], GOLDS["fold1"], SPANS["none"]["fold1"]
for did in G:
    _, bad = gold_to_token_tags(S[did], G[did], LABELS)
    for s, e, lab, *_ in bad[:2]:
        toks = [T[did][a:b] for a, b in S[did] if a < e + 3 and b > s - 1]
        print(f"{lab:10s} gold={T[did][s:e]!r:45s} tokens={toks}")
        shown += 1
    if shown >= 20:
        break


## 9. Tokens and sliding windows per document

In [ ]:
report["windows"] = {}
for m in MODES:
    for split in ("fold1", "fold2", "test"):
        n = sorted(v for k, v in NTOK[m].items() if k.startswith(split + ":"))
        row = dict(tokens_min=n[0], tokens_median=n[len(n)//2], tokens_max=n[-1])
        for w, s in WINDOW_SETTINGS:
            row[f"windows_{w}_{s}"] = sum(len(make_windows(x, w, s)) for x in n)
        report["windows"][f"{m}:{split}"] = row
        print(f"{m:12s} {split:6s}", row)


## 10. Official evaluator: gold vs gold must be 1.0; oracle must match our scorer

In [ ]:
%%writefile /kaggle/working/evaluate_phase2.py
#!/usr/bin/env python3
"""Local evaluator for Phase 2.

Usage:
  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt
"""
import argparse, json, sys

class DuplicateJSONKeyError(ValueError): pass
def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")
def pairs_no_duplicates(pairs):
    out={}; dup=[]
    for k,v in pairs:
        if k in out: dup.append(k)
        out[k]=v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out
def load_json(path):
    with open(path,encoding='utf-8') as f:
        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e
def labels(path):
    with open(path,encoding='utf-8') as f: xs=[x.strip() for x in f if x.strip()]
    if not xs: raise ValueError('labels.txt is empty.')
    if len(xs)!=12: raise ValueError(f'Expected 12 labels, found {len(xs)}.')
    if len(set(xs))!=len(xs): raise ValueError('labels.txt contains duplicate labels.')
    return xs
def spans(x, allowed, where):
    if x is None: return set()
    if not isinstance(x,list): raise ValueError(f'{where}: expected a list of entities.')
    out=set()
    for j,s in enumerate(x):
        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].')
        a,b,l=s[:3]
        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f'{where}, entity {j}: start/end must be integers.')
        if a<0 or b<=a: raise ValueError(f'{where}, entity {j}: require 0 <= start < end.')
        if not isinstance(l,str) or l not in allowed: raise ValueError(f'{where}, entity {j}: invalid label {l!r}.')
        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f'{where}, entity {j}: entity_text must be a string.')
        t=(a,b,l)
        if t in out: raise ValueError(f'{where}, entity {j}: duplicate span {t}.')
        out.add(t)
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 2 entity predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); ap.add_argument('--labels',required=True)
    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)
    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError('Truth and prediction must be non-empty JSON objects.')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    stats={l:[0,0,0] for l in labs}; allowed=set(labs)
    for did in truth:
        g=spans(truth[did],allowed,f'truth[{did}]'); p=spans(pred[did],allowed,f'prediction[{did}]')
        for l in labs:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    precisions=[]; recalls=[]; f1s=[]
    print(f'Examples: {len(truth)}')
    for l in labs:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0
        precisions.append(pr); recalls.append(rc); f1s.append(f)
        print(f'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}')
    print(f'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}')
    print(f'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}')
    print(f'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


In [ ]:
def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def validate_prediction(pred, texts, labels):
    """The checks Codabench applies: same ids, list of [start,end,label], ints, 0<=start<end<=len(text), known label, no duplicates."""
    assert set(pred) == set(texts), f"id mismatch: missing {len(set(texts)-set(pred))}, extra {len(set(pred)-set(texts))}"
    for did, ents in pred.items():
        assert isinstance(ents, list)
        seen = set()
        for x in ents:
            assert isinstance(x, list) and len(x) in (3, 4)
            s, e, l = x[:3]
            assert type(s) is int and type(e) is int and 0 <= s < e <= len(texts[did]) and l in labels, (did, x)
            assert (s, e, l) not in seen, ("duplicate", did, x); seen.add((s, e, l))
    return True

gold_f1 = {k: [x[:3] for x in v] for k, v in GOLDS["fold1"].items()}
write_json(GOLDS["fold1"], f"{WORK}/truth_fold1.json")
write_json(gold_f1, f"{WORK}/pred_gold_fold1.json")
pred_oracle, s_oracle = ORACLE_FOLD1_NONE
validate_prediction(pred_oracle, TEXTS["fold1"], LABELS)
write_json(pred_oracle, f"{WORK}/pred_oracle_fold1.json")
with open(f"{WORK}/labels.txt", "w") as f:
    f.write("\n".join(LABELS) + "\n")
for name in ("pred_gold_fold1.json", "pred_oracle_fold1.json"):
    r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase2.py", "--truth", f"{WORK}/truth_fold1.json",
                        "--prediction", f"{WORK}/{name}", "--labels", f"{WORK}/labels.txt"], capture_output=True, text=True)
    print(f"--- {name}\n" + r.stdout.strip().splitlines()[-3] if r.returncode == 0 else r.stderr)
print(f"our scorer on the oracle: {s_oracle['macro_F1']:.6f}  (must equal the official Strict Macro-F1 line above)")

empty_pred = {k: [] for k in TEST_TEXTS}
validate_prediction(empty_pred, TEST_TEXTS, LABELS)
print("format validator OK on an all-empty fold3 prediction (including the empty document). NOT a submission.")


## 11. Save the report (for the final write-up) and upload it to the bucket

In [ ]:
report["tokenizer"] = TOKENIZER_NAME
report["created"] = time.strftime("%Y-%m-%d %H:%M:%S")
write_json(report, REPORT_PATH)
try:
    upload_model_file(REPORT_PATH)
except Exception as e:
    print("upload failed (the file is still in /kaggle/working):", type(e).__name__, e)
print(json.dumps({m: {f: (report['oracle'][m][f]['ceiling_macroF1_no_repair'], report['oracle'][m][f]['ceiling_macroF1_with_repair'])
                      for f in TEXTS} for m in MODES}, indent=1))


## 12. What to read from this, and the decision it feeds

- **Section 7** must say `OK`. If it fails, stop and send me the error: the offsets would be wrong for every model.
- **Section 8** is the key table. It is the best strict macro-F1 any *token-level* model could reach with each input mode:
  - `none` keeps the Phase-1 tokens exactly (best transfer from pretraining), but many entity ends sit inside a token like `Court,`; the learned repair recovers most of them.
  - `punct` / `punct_digit` make almost every boundary reachable, at the price of more tokens per word and token sequences the Phase-1 models never saw (`Court</w>` `,</w>` instead of `Court,</w>`).
  - Choosing the mode (or trying both, as a preprocessing experiment for the report) is your decision; the later notebooks will take it as a setting.
- **Section 9** tells you how many windows each document needs, i.e. the cost of training/inference at each window size.
- **Section 10**: the official evaluator must print `Strict Macro-F1: 1.000000` for gold-vs-gold, and the oracle line must equal our scorer.

Send me the printed output of sections 7–10 (or the `phase2_alignment_report.json`).